# 04 Mutual Information Extension

Optional nonlinear dependence experiment. This notebook uses a smaller node subset because pairwise mutual information is expensive.

In [1]:
from pathlib import Path
import sys

import networkx as nx
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import EDGE_DENSITY, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, RANDOM_SEED
from src.dependence import compute_mutual_information_matrix
from src.network_construction import adjacency_from_fixed_density, build_graph_from_adjacency

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
X = np.load(INTERIM_DATA_DIR / "X_preprocessed.npy")
node_metadata = pd.read_parquet(INTERIM_DATA_DIR / "node_metadata.parquet")

max_nodes = 200
n_subset = min(max_nodes, X.shape[1])
rng = np.random.default_rng(RANDOM_SEED)
subset = np.sort(rng.choice(X.shape[1], size=n_subset, replace=False))

X_subset = X[:, subset]
metadata_subset = node_metadata.iloc[subset].copy().reset_index(drop=True)
metadata_subset["original_filtered_node_id"] = metadata_subset["node_id"].to_numpy()
metadata_subset["node_id"] = np.arange(n_subset)

print(f"MI subset matrix shape: {X_subset.shape}")

MI subset matrix shape: (612, 200)


In [3]:
mi = compute_mutual_information_matrix(X_subset, n_bins=16)
np.save(PROCESSED_DATA_DIR / "mutual_information_subset.npy", mi)

A_mi = adjacency_from_fixed_density(mi, density=EDGE_DENSITY, use_absolute=False)
np.save(PROCESSED_DATA_DIR / "mutual_information_subset_adjacency.npy", A_mi)

G_mi = build_graph_from_adjacency(A_mi, metadata_subset)
nx.write_graphml(G_mi, PROCESSED_DATA_DIR / "mutual_information_subset_network.graphml")

print(G_mi)
print(f"Actual edge density: {nx.density(G_mi):.4f}")

Mutual information rows: 100%|██████████| 200/200 [00:06<00:00, 29.22it/s]

Graph with 200 nodes and 199 edges
Actual edge density: 0.0100
